# boolean-mask-combine composite — cx26: combine masks via repeat-broadcast pairing — every ray vs every triangle

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `boolean-mask-combine`, `einops-repeat-broadcast`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "boolean-mask-combine"
DD_ATOM_IDS = ["boolean-mask-combine", "einops-repeat-broadcast"]
DD_SUBTOPICS = ["Numpy: Boolean mask combine", "Einops: Repeat-as-broadcast"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Same problem shape as cx25 but a different mechanic. `einops.repeat` with an axis name that matches the SOURCE shape produces a **stride-0 view** — no copy, no materialized tensor — just a broadcast view sized for the joint `(NR, NT)` grid. This is the einops-native version of `tensor.expand(...)`.

**The pairing pattern.** When you want every ray to be paired with every triangle, you `repeat` *both* per-axis tensors onto the joint grid with stride-0 views, then `&` them. The result is a real boolean tensor — but the inputs to `&` were free.

**Anatomy.**
- `repeat(ray_ok, 'r -> r t', t=NT)` — `(NR,)` lifted to `(NR, NT)` via stride-0 broadcast on `t`.
- `repeat(tri_ok, 't -> r t', r=NR)` — `(NT,)` lifted to `(NR, NT)` via stride-0 broadcast on `r`.
- `&` — materializes the boolean grid, ONE allocation total.

**Why care.** In ARENA the rays-vs-triangles cross product is the inner loop. A stride-0 lift keeps memory traffic linear in `NR + NT`, not `NR * NT`, up until the final AND.

### Composite Exercise — combine masks via repeat-broadcast pairing — every ray vs every triangle

**Atoms exercised together**: `boolean-mask-combine`, `einops-repeat-broadcast`

Implement `cx26_pair_masks_broadcast(ray_ok, tri_ok)` — same I/O contract as cx25, but THIS drill exercises the broadcast/no-copy property of einops repeat.

- `ray_ok`: boolean tensor of shape `(NR,)`.
- `tri_ok`: boolean tensor of shape `(NT,)`.
- Return: boolean tensor of shape `(NR, NT)` matching `ray_ok[:, None] & tri_ok[None, :]`.

1. **Repeat-broadcast** — use `einops.repeat` to lift both masks. Pre-AND, the two views must be stride-0 along their broadcast axis (no copy).
2. **Boolean combine** — `&` to materialize the joint grid.

The test verifies the values AND probes the stride-0 property of the intermediate broadcasts.

In [ ]:
def cx26_lift_only(ray_ok, tri_ok):
    NR = ray_ok.shape[0]
    NT = tri_ok.shape[0]
    # einops repeat with a name-only target axis = stride-0 broadcast view, no copy.
    ray_grid = repeat(ray_ok, 'r -> r t', t=NT)
    tri_grid = repeat(tri_ok, 't -> r t', r=NR)
    return ray_grid, tri_grid

def cx26_pair_masks_broadcast(ray_ok, tri_ok):
    ray_grid, tri_grid = cx26_lift_only(ray_ok, tri_ok)
    # Atom (boolean-mask-combine): ONE materializing op on the joint shape.
    return ray_grid & tri_grid


<details><summary>Show solution — cx26</summary>

```python
def cx26_lift_only(ray_ok, tri_ok):
    NR = ray_ok.shape[0]
    NT = tri_ok.shape[0]
    # einops repeat with a name-only target axis = stride-0 broadcast view, no copy.
    ray_grid = repeat(ray_ok, 'r -> r t', t=NT)
    tri_grid = repeat(tri_ok, 't -> r t', r=NR)
    return ray_grid, tri_grid

def cx26_pair_masks_broadcast(ray_ok, tri_ok):
    ray_grid, tri_grid = cx26_lift_only(ray_ok, tri_ok)
    # Atom (boolean-mask-combine): ONE materializing op on the joint shape.
    return ray_grid & tri_grid
```

Note both helper and combine return the same answer as cx25, but the intermediate tensors are stride-0 views — proving the repeat was a broadcast, not a tile. The `& ` is the only place memory is allocated for the full `(NR, NT)` grid.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx26'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx26',
        'subtopics': ["Numpy: Boolean mask combine", "Einops: Repeat-as-broadcast"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()